# Praktikum 3 — Implementasi Algoritma Pencarian Buta dengan Python

**Mata Kuliah:** Kecerdasan Buatan (RPL3404) · **Program Studi:** D4 Teknologi Rekayasa Perangkat Lunak
**Jurusan Teknologi Informasi — Politeknik Negeri Padang** · Buku Ajar Bab 3: *Algoritma Pencarian Buta*

| Identitas | Isian |
|---|---|
| Nama | Windi SRi Rahayu |
| NIM | 2611089002 |
| Kelas | TRPL AK 1|
| Tanggal praktikum | 29 September 2026 |
| Tautan repositori Git | |

> Jalankan sel secara berurutan dari atas ke bawah (**Runtime → Run all** di Colab atau **Run All** di Jupyter/VS Code).

## A. Tujuan Praktikum
1. Merepresentasikan masalah pencarian (`Problem`) dan node pencarian (`Node`) sebagai kelas Python.
2. Mengimplementasikan BFS, UCS, DFS, depth-limited search (DLS), dan iterative deepening search (IDS).
3. Membandingkan rute, jumlah langkah, biaya, jumlah node yang di-expand, dan waktu eksekusi setiap algoritma.
4. Menerapkan antarmuka `Problem` yang sama pada masalah lain (8-puzzle) untuk menunjukkan generalitas algoritma.

## B. Alat dan Bahan
- Python 3.10+ (pustaka standar: `collections`, `heapq`, `time`)
- Opsional untuk visualisasi dan tabel: `networkx`, `matplotlib`, `pandas` (sudah tersedia di Google Colab)
- Buku Ajar Bab 3, terutama Gambar 3.1 dan Tabel 3.1–3.5

## C. Alokasi Waktu (2 jam praktik)
| Kegiatan | Durasi |
|---|---|
| Pengantar dan penyiapan lingkungan | 10 menit |
| Langkah 1–3: representasi masalah, BFS, UCS | 35 menit |
| Langkah 4–5: DFS, DLS, IDS | 30 menit |
| Langkah 6–7: eksperimen perbandingan dan 8-puzzle | 35 menit |
| Diskusi dan penyusunan kesimpulan | 10 menit |

In [ ]:
# Cek versi Python dan pustaka opsional
import sys
print("Python", sys.version.split()[0])
try:
    import networkx, matplotlib, pandas
    print("networkx", networkx.__version__, "| matplotlib", matplotlib.__version__, "| pandas", pandas.__version__)
except ImportError as e:
    print("Pustaka opsional belum terpasang:", e, "-> jalankan: pip install networkx matplotlib pandas")

## D. Langkah Kerja

### Langkah 1 — Representasi masalah
Graph peta disimpan sebagai *dictionary* bersarang: kunci luar adalah kota, nilai dalam adalah jarak (km) ke kota tetangga.
Kelas `Problem` membungkus komponen formal masalah pencarian (Tabel 3.1): **initial state, goal, ACTIONS, RESULT, ACTION-COST**.
Kelas `Node` menyimpan **STATE, PARENT, ACTION, PATH-COST** ditambah `depth` (Subbab 3.3.2).

> **Catatan:** urutan tetangga pada dictionary menentukan urutan node dibangkitkan. Jarak dibulatkan untuk ilustrasi, bukan data resmi.

In [ ]:
from collections import deque
import heapq

PETA_SUMBAR = {
    "Padang":         {"Pariaman": 56, "Padang Panjang": 72, "Solok": 64, "Painan": 77},
    "Pariaman":       {"Padang": 56, "Lubuk Basung": 48, "Padang Panjang": 50},
    "Painan":         {"Padang": 77},
    "Lubuk Basung":   {"Pariaman": 48, "Bukittinggi": 56},
    "Padang Panjang": {"Padang": 72, "Pariaman": 50, "Batusangkar": 36, "Bukittinggi": 19},
    "Bukittinggi":    {"Padang Panjang": 19, "Lubuk Basung": 56, "Payakumbuh": 33},
    "Batusangkar":    {"Padang Panjang": 36, "Solok": 55, "Sawahlunto": 46, "Payakumbuh": 42},
    "Solok":          {"Padang": 60, "Batusangkar": 55, "Sawahlunto": 36},
    "Sawahlunto":     {"Solok": 36, "Batusangkar": 46},
    "Payakumbuh":     {"Bukittinggi": 33, "Batusangkar": 42},
}

class Problem:
    """Masalah pencarian: initial state, goal, ACTIONS, RESULT, ACTION-COST."""
    def __init__(self, initial, goal, graph):
        self.initial, self.goal, self.graph = initial, goal, graph
    def actions(self, s):            # aksi = pindah ke kota tetangga
        return list(self.graph[s])
    def result(self, s, a):          # transition model
        return a
    def action_cost(self, s, a, s2):
        return self.graph[s][s2]
    def is_goal(self, s):
        return s == self.goal

class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state, self.parent = state, parent
        self.action, self.path_cost = action, path_cost
        self.depth = 0 if parent is None else parent.depth + 1
    def __lt__(self, other):          # dibutuhkan heapq saat biaya sama
        return self.state < other.state

def expand(problem, node):
    s = node.state
    for a in problem.actions(s):
        s2 = problem.result(s, a)
        cost = node.path_cost + problem.action_cost(s, a, s2)
        yield Node(s2, node, a, cost)

def path_states(node):
    path = []
    while node is not None:
        path.append(node.state)
        node = node.parent
    return path[::-1]

Sebelum menjalankan algoritma, uji dulu representasi masalah. Pastikan `actions`, `result`, dan `action_cost` bekerja sesuai contoh pada Tabel 3.1.

In [ ]:
masalah = Problem("Padang", "Payakumbuh", PETA_SUMBAR)

print("ACTIONS(Padang)            =", masalah.actions("Padang"))
print("RESULT(Padang, Solok)      =", masalah.result("Padang", "Solok"))
print("ACTION-COST(Padang, Solok) =", masalah.action_cost("Padang", "Solok", "Solok"), "km")
print("IS-GOAL(Payakumbuh)        =", masalah.is_goal("Payakumbuh"))

# Cek bahwa setiap jalan bersifat dua arah dengan jarak yang sama
for a, tetangga in PETA_SUMBAR.items():
    for b, w in tetangga.items():
        assert PETA_SUMBAR[b][a] == w, f"Jarak {a}-{b} tidak simetris"
print("Graph valid: semua jalan dua arah dan simetris.")

**Visualisasi state space (Gambar 3.1).** Sel ini opsional; lewati jika `networkx`/`matplotlib` tidak tersedia.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

POSISI = {"Lubuk Basung": (0.2, 4.2), "Pariaman": (0.6, 2.6), "Padang": (1.4, 1.0),
          "Painan": (2.6, -0.4), "Padang Panjang": (2.6, 2.9), "Bukittinggi": (2.3, 4.4),
          "Payakumbuh": (4.9, 4.6), "Batusangkar": (4.2, 3.0), "Solok": (3.8, 1.3),
          "Sawahlunto": (5.4, 1.9)}

def gambar_peta(graph, jalur=None, judul="Peta jalan Sumatera Barat (disederhanakan)"):
    G = nx.Graph()
    for a, nb in graph.items():
        for b, w in nb.items():
            G.add_edge(a, b, weight=w)
    fig, ax = plt.subplots(figsize=(9, 6))
    nx.draw_networkx_edges(G, POSISI, ax=ax, width=1.5, edge_color="#718096")
    if jalur:
        sisi = list(zip(jalur, jalur[1:]))
        nx.draw_networkx_edges(G, POSISI, edgelist=sisi, ax=ax, width=4, edge_color="#DD6B20")
    nx.draw_networkx_edge_labels(G, POSISI, edge_labels=nx.get_edge_attributes(G, "weight"), ax=ax, font_size=9)
    warna = ["#B7E4C7" if n == "Padang" else "#FFD166" if n == "Payakumbuh" else "#E2E8F0" for n in G]
    nx.draw_networkx_nodes(G, POSISI, node_color=warna, node_size=700, edgecolors="#2D3748", ax=ax)
    nx.draw_networkx_labels(G, {k: (x, y + 0.38) for k, (x, y) in POSISI.items()}, font_size=9, font_weight="bold", ax=ax)
    ax.set_title(judul)
    ax.axis("off")
    plt.show()

gambar_peta(PETA_SUMBAR)

### Langkah 2 — Breadth-First Search (BFS)
BFS memakai **FIFO queue** (`deque`) dan himpunan `reached`. Goal diperiksa saat node dibangkitkan (**early goal test**).
Parameter `stats` adalah dictionary untuk menghitung jumlah node yang di-expand.

In [ ]:
def breadth_first_search(problem, stats):
    node = Node(problem.initial)
    if problem.is_goal(node.state):
        return node
    frontier = deque([node])                 # FIFO queue
    reached = {problem.initial}
    while frontier:
        node = frontier.popleft()
        stats["expanded"] += 1
        for child in expand(problem, node):
            s = child.state
            if problem.is_goal(s):           # early goal test
                return child
            if s not in reached:
                reached.add(s)
                frontier.append(child)
    return None

In [ ]:
stats = {"expanded": 0}
hasil = breadth_first_search(masalah, stats)
print("Rute   :", " -> ".join(path_states(hasil)))
print("Langkah:", hasil.depth, "| Biaya:", hasil.path_cost, "km | Ekspansi:", stats["expanded"])

**Penelusuran BFS (bandingkan dengan Tabel 3.3).** Versi berikut sama dengan BFS di atas, tetapi mencetak isi frontier setiap iterasi.

In [ ]:
def bfs_trace(problem):
    node = Node(problem.initial)
    frontier = deque([node])
    reached = {problem.initial}
    iterasi = 0
    while frontier:
        node = frontier.popleft()
        iterasi += 1
        baru = []
        for child in expand(problem, node):
            if problem.is_goal(child.state):
                print(f"{iterasi:>2}. expand {node.state:<15} -> GOAL {child.state}")
                return child
            if child.state not in reached:
                reached.add(child.state)
                frontier.append(child)
                baru.append(child.state)
        print(f"{iterasi:>2}. expand {node.state:<15} tambah {baru}")
        print(f"    frontier: {[n.state for n in frontier]}")
    return None

solusi = bfs_trace(masalah)
print("Solusi:", " -> ".join(path_states(solusi)))

### Langkah 3 — Uniform-Cost Search (UCS)
UCS memakai **priority queue** (`heapq`) yang diurutkan menurut path cost `g(n)`. Karena `heapq` tidak mendukung "perbarui prioritas",
node dengan biaya baru langsung dimasukkan; entri lama yang lebih mahal dilewati saat diambil (*lazy deletion*).
Goal diperiksa saat node diambil dari frontier (**late goal test**).

In [ ]:
def uniform_cost_search(problem, stats):
    node = Node(problem.initial)
    frontier = [(0, node)]                   # priority queue (heap)
    reached = {problem.initial: node}
    while frontier:
        g, node = heapq.heappop(frontier)
        if g > reached[node.state].path_cost:
            continue                         # entri usang, lewati
        if problem.is_goal(node.state):      # late goal test
            return node
        stats["expanded"] += 1
        for child in expand(problem, node):
            s = child.state
            if s not in reached or child.path_cost < reached[s].path_cost:
                reached[s] = child
                heapq.heappush(frontier, (child.path_cost, child))
    return None

In [ ]:
stats = {"expanded": 0}
hasil = uniform_cost_search(masalah, stats)
print("Rute   :", " -> ".join(path_states(hasil)))
print("Langkah:", hasil.depth, "| Biaya:", hasil.path_cost, "km | Ekspansi:", stats["expanded"])

**Penelusuran UCS (bandingkan dengan Tabel 3.4).** Perhatikan kapan biaya Batusangkar diperbarui dan kapan Payakumbuh diakui sebagai goal.

In [ ]:
def ucs_trace(problem):
    node = Node(problem.initial)
    frontier = [(0, node)]
    reached = {problem.initial: node}
    iterasi = 0
    while frontier:
        g, node = heapq.heappop(frontier)
        if g > reached[node.state].path_cost:
            print(f"    (lewati entri usang {node.state} = {g})")
            continue
        if problem.is_goal(node.state):
            print(f"{iterasi + 1:>2}. ambil {node.state} ({g}) -> GOAL")
            return node
        iterasi += 1
        catatan = []
        for child in expand(problem, node):
            s = child.state
            if s not in reached or child.path_cost < reached[s].path_cost:
                if s in reached:
                    catatan.append(f"    perbarui {s}: {reached[s].path_cost} -> {child.path_cost}")
                reached[s] = child
                heapq.heappush(frontier, (child.path_cost, child))
        isi = sorted((c, n.state) for c, n in frontier if c <= reached[n.state].path_cost)
        print(f"{iterasi:>2}. expand {node.state} ({g}) | frontier: " + ", ".join(f"{s} {c}" for c, s in isi))
        for teks in catatan:
            print(teks)
    return None

solusi = ucs_trace(masalah)
print("Solusi:", " -> ".join(path_states(solusi)), "=", solusi.path_cost, "km")

### Langkah 4 — Depth-First Search (DFS)
DFS diimplementasikan sebagai **tree-like search** dengan `list` sebagai stack (LIFO). Fungsi `is_cycle` menelusuri rantai parent
untuk mencegah loop (pendekatan ketiga pada Subbab 3.3.3).

In [ ]:
def is_cycle(node):
    s, p = node.state, node.parent
    while p is not None:
        if p.state == s:
            return True
        p = p.parent
    return False

def depth_first_search(problem, stats):
    frontier = [Node(problem.initial)]       # LIFO queue (stack)
    while frontier:
        node = frontier.pop()
        if problem.is_goal(node.state):
            return node
        stats["expanded"] += 1
        children = [c for c in expand(problem, node) if not is_cycle(c)]
        frontier.extend(reversed(children))  # tetangga pertama diproses dulu
    return None

In [ ]:
stats = {"expanded": 0}
hasil = depth_first_search(masalah, stats)
print("Rute   :", " -> ".join(path_states(hasil)))
print("Langkah:", hasil.depth, "| Biaya:", hasil.path_cost, "km | Ekspansi:", stats["expanded"])

### Langkah 5 — Depth-Limited Search (DLS) dan Iterative Deepening Search (IDS)
DLS mengembalikan node solusi, `None` (*failure*), atau `"cutoff"`. IDS memanggil DLS dengan batas kedalaman yang terus bertambah.

In [ ]:
CUTOFF = "cutoff"

def depth_limited_search(problem, limit, stats):
    frontier = [Node(problem.initial)]
    result = None                            # None berarti failure
    while frontier:
        node = frontier.pop()
        if problem.is_goal(node.state):
            return node
        if node.depth >= limit:
            result = CUTOFF
        elif not is_cycle(node):
            stats["expanded"] += 1
            frontier.extend(reversed(list(expand(problem, node))))
    return result

def iterative_deepening_search(problem, stats):
    depth = 0
    while True:
        result = depth_limited_search(problem, depth, stats)
        if result != CUTOFF:
            return result
        depth += 1

In [ ]:
# Amati hasil DLS untuk setiap batas kedalaman (Subbab 3.4.4)
for batas in range(5):
    st = {"expanded": 0}
    r = depth_limited_search(masalah, batas, st)
    keterangan = r if r in (None, CUTOFF) else " -> ".join(path_states(r))
    print(f"limit={batas}: {keterangan} | ekspansi={st['expanded']}")

stats = {"expanded": 0}
hasil = iterative_deepening_search(masalah, stats)
print("\nIDS:", " -> ".join(path_states(hasil)), "| biaya", hasil.path_cost, "km | total ekspansi", stats["expanded"])

### Langkah 6 — Eksperimen perbandingan
Fungsi `jalankan` menjalankan satu algoritma dan mencetak rute, langkah, biaya, jumlah ekspansi, serta waktu.
Nilai waktu dapat berbeda pada setiap komputer.

In [ ]:
import time

def jalankan(nama, fungsi, problem):
    stats = {"expanded": 0}
    t0 = time.perf_counter()
    hasil = fungsi(problem, stats)
    dt = (time.perf_counter() - t0) * 1000
    if hasil is None:
        print(f"{nama:<5} gagal menemukan solusi")
        return
    rute = " -> ".join(path_states(hasil))
    print(f"{nama:<5} | {rute}")
    print(f"      | langkah={hasil.depth}, biaya={hasil.path_cost} km, "
          f"ekspansi={stats['expanded']}, waktu={dt:.3f} ms")

masalah = Problem("Padang", "Payakumbuh", PETA_SUMBAR)
for nama, f in [("BFS", breadth_first_search),
                ("UCS", uniform_cost_search),
                ("DFS", depth_first_search),
                ("IDS", iterative_deepening_search)]:
    jalankan(nama, f, masalah)

In [ ]:
# Ringkasan dalam bentuk tabel (bandingkan dengan Tabel 3.8)
import pandas as pd

ALGORITMA = [("BFS", breadth_first_search), ("UCS", uniform_cost_search),
             ("DFS", depth_first_search), ("IDS", iterative_deepening_search)]

def bandingkan(problem, algoritma=ALGORITMA, ulang=200):
    baris = []
    for nama, f in algoritma:
        t0 = time.perf_counter()
        for _ in range(ulang):                      # diulang agar waktu lebih stabil
            stats = {"expanded": 0}
            hasil = f(problem, stats)
        dt = (time.perf_counter() - t0) / ulang * 1000
        baris.append({"Algoritma": nama,
                      "Rute": " - ".join(path_states(hasil)) if hasil else "-",
                      "Langkah": hasil.depth if hasil else None,
                      "Biaya (km)": hasil.path_cost if hasil else None,
                      "Node di-expand": stats["expanded"],
                      "Waktu rata-rata (ms)": round(dt, 4)})
    return pd.DataFrame(baris)

tabel = bandingkan(masalah)
tabel

In [ ]:
# Visualisasi: biaya dan jumlah ekspansi tiap algoritma
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(tabel["Algoritma"], tabel["Biaya (km)"], color="#4C78A8")
axes[0].set_title("Biaya solusi (km)")
axes[1].bar(tabel["Algoritma"], tabel["Node di-expand"], color="#F58518")
axes[1].set_title("Jumlah node di-expand")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

# Tampilkan rute BFS dan UCS pada peta
for nama, f in [("BFS", breadth_first_search), ("UCS", uniform_cost_search)]:
    h = f(masalah, {"expanded": 0})
    gambar_peta(PETA_SUMBAR, path_states(h), f"Rute {nama}: {h.path_cost} km")

**Pertanyaan analisis Langkah 6** (jawab di sel Markdown di bawah):
1. Mengapa BFS dan UCS menghasilkan rute berbeda walaupun jumlah langkahnya sama?
2. Mengapa DFS menghasilkan rute yang jauh lebih panjang?
3. Mengapa IDS meng-expand lebih banyak node daripada BFS, tetapi tetap dianjurkan untuk state space besar?

*Jawaban:*

### Langkah 7 — Menerapkan algoritma yang sama pada 8-puzzle
Kelas `EightPuzzle` mewarisi `Problem` dan hanya mengganti `actions`, `result`, dan `action_cost`.
BFS dan IDS dipakai **tanpa perubahan**. State adalah tuple 9 angka dengan `0` sebagai ubin kosong.

In [ ]:
class EightPuzzle(Problem):
    GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)          # 0 = ubin kosong
    GESER = {"Atas": -3, "Bawah": 3, "Kiri": -1, "Kanan": 1}
    def __init__(self, initial):
        self.initial, self.goal = tuple(initial), self.GOAL
    def actions(self, s):
        i = s.index(0)
        aksi = []
        if i >= 3: aksi.append("Atas")
        if i <= 5: aksi.append("Bawah")
        if i % 3 > 0: aksi.append("Kiri")
        if i % 3 < 2: aksi.append("Kanan")
        return aksi
    def result(self, s, a):
        i = s.index(0)
        j = i + self.GESER[a]
        s = list(s)
        s[i], s[j] = s[j], s[i]
        return tuple(s)
    def action_cost(self, s, a, s2):
        return 1

def cetak_papan(s):
    for r in range(3):
        print(" ".join("_" if v == 0 else str(v) for v in s[3*r:3*r+3]))

In [ ]:
kasus = {"mudah":  (1, 2, 3, 4, 0, 6, 7, 5, 8),
         "sedang": (4, 1, 3, 7, 2, 6, 0, 5, 8),
         "sulit":  (3, 5, 0, 1, 4, 2, 7, 8, 6),
         "berat":  (1, 8, 3, 5, 0, 6, 4, 2, 7)}

baris = []
for nama, awal in kasus.items():
    for label, f in [("BFS", breadth_first_search), ("IDS", iterative_deepening_search)]:
        stats = {"expanded": 0}
        t0 = time.perf_counter()
        hasil = f(EightPuzzle(awal), stats)
        dt = (time.perf_counter() - t0) * 1000
        print(f"{nama:<7}{label}: langkah={hasil.depth}, ekspansi={stats['expanded']}")
        baris.append({"Kasus": nama, "Algoritma": label, "Langkah": hasil.depth,
                      "Node di-expand": stats["expanded"], "Waktu (ms)": round(dt, 2)})

tabel_puzzle = pd.DataFrame(baris)
tabel_puzzle

In [ ]:
# Tampilkan langkah-langkah solusi untuk kasus "sedang"
hasil = breadth_first_search(EightPuzzle(kasus["sedang"]), {"expanded": 0})
node, urutan = hasil, []
while node is not None:
    urutan.append(node)
    node = node.parent
for i, n in enumerate(reversed(urutan)):
    print(f"Langkah {i}: {n.action or 'awal'}")
    cetak_papan(n.state)
    print()

### Pengayaan — Pertumbuhan eksponensial node (Subbab 3.4.1 dan 3.4.4)
Rumus jumlah node yang dibangkitkan pada pohon seragam dengan branching factor *b* dan kedalaman solusi *d*:

- $N(BFS) = b + b^2 + \dots + b^d$
- $N(IDS) = (d)b + (d-1)b^2 + \dots + (1)b^d$

In [ ]:
def ukuran(kb):
    for satuan in ["KB", "MB", "GB", "TB", "PB"]:
        if kb < 1000:
            return f"{kb:,.1f} {satuan}"
        kb /= 1000
    return f"{kb:,.1f} EB"

def n_bfs(b, d):
    return sum(b**i for i in range(1, d + 1))

def n_ids(b, d):
    return sum((d - i + 1) * b**i for i in range(1, d + 1))

print("b=10, d=5 -> N(IDS) =", f"{n_ids(10, 5):,}", "| N(BFS) =", f"{n_bfs(10, 5):,}")

baris = []
for d in range(2, 15, 2):
    bfs_n, ids_n = n_bfs(10, d), n_ids(10, d)
    baris.append({"d": d, "N(BFS)": bfs_n, "N(IDS)": ids_n,
                  "Tambahan IDS (%)": round((ids_n - bfs_n) / bfs_n * 100, 1),
                  "Memori BFS (1 KB/node)": ukuran(bfs_n)})
pd.DataFrame(baris)

## E. Tugas Praktikum
Kerjakan setiap tugas di sel yang disediakan. Tambahkan sel baru bila perlu.

### Tugas 1
Ubah initial state dan goal menjadi **Painan → Lubuk Basung**. Jalankan keempat algoritma dan buat tabel perbandingan seperti Tabel 3.8.

In [ ]:
# TODO Tugas 1
masalah2 = Problem("Painan", "Lubuk Basung", PETA_SUMBAR)
# bandingkan(masalah2)

### Tugas 2
Ubah urutan tetangga **Padang Panjang** sehingga `Bukittinggi` berada sebelum `Batusangkar`. Jalankan ulang BFS dan DFS, lalu jelaskan mengapa hasilnya berubah atau tidak berubah.

In [ ]:
# TODO Tugas 2
import copy
PETA_BARU = copy.deepcopy(PETA_SUMBAR)
PETA_BARU["Padang Panjang"] = {"Padang": 72, "Pariaman": 50, "Bukittinggi": 19, "Batusangkar": 36}
masalah3 = Problem("Padang", "Payakumbuh", PETA_BARU)
# jalankan BFS dan DFS pada masalah3

*Penjelasan Tugas 2:*

### Tugas 3
Tambahkan pengukuran **ukuran frontier maksimum** (`stats["max_frontier"]`) pada setiap algoritma sebagai pendekatan space complexity.
Bandingkan hasil BFS dan IDS pada kasus 8-puzzle "berat".

In [ ]:
# TODO Tugas 3
# Petunjuk: setelah frontier bertambah, perbarui
#   stats["max_frontier"] = max(stats.get("max_frontier", 0), len(frontier))

### Tugas 4
Implementasikan versi DFS yang menggunakan tabel `reached` (**graph search**). Bandingkan jumlah node yang di-expand dengan versi tree-like search.

In [ ]:
# TODO Tugas 4
def depth_first_graph_search(problem, stats):
    raise NotImplementedError

### Tugas 5 (Pengayaan)
Implementasikan **bidirectional breadth-first search** pada peta Sumatera Barat dan bandingkan jumlah node yang di-expand dengan BFS biasa.

In [ ]:
# TODO Tugas 5
def bidirectional_bfs(graph, start, goal, stats):
    raise NotImplementedError

### Tugas 6 (Pengayaan)
Bangun graph lokasi di lingkungan kampus PNP (minimal 12 node, misalnya gerbang, gedung jurusan, perpustakaan, masjid, kantin, parkiran)
dengan estimasi jarak berjalan kaki (meter). Gunakan UCS untuk mencari rute terpendek antara dua lokasi.

In [ ]:
# TODO Tugas 6
KAMPUS_PNP = {
    # "Gerbang Utama": {"Parkiran": 80, ...},
}

## F. Kesimpulan
Tuliskan kesimpulan praktikum dengan mengaitkan hasil empiris pada sel-sel di atas dengan sifat **completeness, cost optimality, time complexity,
dan space complexity** pada Tabel 3.5 Buku Ajar.

*Kesimpulan:*

---
**Pernyataan penggunaan AI generatif** (isi jika ada, sesuai ketentuan RPS):

**Rujukan:** Russell, S. J., & Norvig, P. (2022). *Artificial intelligence: A modern approach* (Ed. ke-4, Global Edition). Pearson. Bab 3, Subbab 3.1–3.4.